In [0]:
%pip install --quiet h2o interpret h2o_pysparkling_3.5
#%%sh pip install --quiet h2o

In [0]:
#dbutils.library.restartPython()

In [0]:
import h2o
from h2o.automl import H2OAutoML
from pysparkling import H2OContext
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import pickle
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pysparkling import H2OContext
from sklearn.impute import KNNImputer
from h2o.automl import H2OAutoML

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [0]:
model_train_data_final = spark.read.csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_trainingdata', header = True)
test_data_final = spark.read.csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_testdata', header = True)

#model_train_data_final_pd = model_train_data_final.toPandas()
#test_data_final_pd = test_data_final.toPandas()

print(f"model_train_data_final row count: {model_train_data_final.count()}")
model_train_data_final.display()
test_data_final.display()

In [0]:
# Ordinal Encoding for Segmentations

def ordinal_transform(df):
    # Use string representations for replacement values
    seg_mapping = {"Unassigned": "0", "L": "1", "M": "2", "H": "3"}
    dig_mapping = {"New": "0", "Unassigned": "1", "L": "2", "M": "3", "H": "4"}

    seg_columns = ["price_dim_seg", "health_dim_seg", "convenience_dim_seg", "variety_seeking_dim_seg"]
    all_target_cols = seg_columns + ["dig_eng_seg_code"]

    for col in all_target_cols:
        df = df.withColumn(col, f.trim(f.col(col)))

    df = df.replace(to_replace=seg_mapping, subset=seg_columns)
    df = df.replace(to_replace=dig_mapping, subset=["dig_eng_seg_code"])
    
    for col in all_target_cols:
        df = df.withColumn(col, f.col(col).cast("integer"))

    return df

In [0]:
model_train_data_final = ordinal_transform(model_train_data_final)
test_data_final = ordinal_transform(test_data_final)

model_train_data_final.display()

In [0]:
# Create H2O instance 
h2o.init() 

In [0]:
# Initialize H2O Context
hc = H2OContext.getOrCreate()

# Convert PySpark DataFrames to H20Frame

# Need to downsample - tooooo many HHs
positives = model_train_data_final.filter(f.col("target_label") == "1")
negatives = model_train_data_final.filter(f.col("target_label") == "0").sample(False, 0.05, seed=8451)
model_train_data_final_downsampled = positives.union(negatives)

h2o_train = hc.asH2OFrame(model_train_data_final_downsampled)
h2o_test = hc.asH2OFrame(test_data_final)


In [0]:
# Convert enum to int for continuous variables
numeric_cols = [
    "total_net_spend", "total_trips", 
    "holiday_2024_kpf_net_spend", "holiday_2024_kpf_trips",
    "spr_eas_mothers_2025_kpf_net_spend", "spr_eas_mothers_2025_kpf_trips",
    "fathers_summer_grad_2025_kpf_net_spend", "fathers_summer_grad_2025_kpf_trips",
    "vtines_2025_kpf_net_spend", "vtines_2025_kpf_trips",
    "fall_2025_kpf_net_spend", "fall_2025_kpf_trips",
    "holiday_2024_kpf_spend_pct", "spr_eas_mothers_2025_kpf_spend_pct",
    "fathers_summer_grad_2025_kpf_spend_pct", "vtines_2025_kpf_spend_pct", "fall_2025_kpf_spend_pct",
    "total_fuel_points", "holiday_2024_kpf_fuel_points", "spr_eas_mothers_2025_kpf_fuel_points",
    "fathers_summer_grad_2025_kpf_fuel_points", "vtines_2025_kpf_fuel_points", "fall_2025_kpf_fuel_points",
    "holiday_2024_kpf_fuel_points_pct", "spr_eas_mothers_2025_kpf_fuel_points_pct",
    "fathers_summer_grad_2025_kpf_fuel_points_pct", "vtines_2025_kpf_fuel_points_pct",
    "fall_2025_kpf_fuel_points_pct", "total_greeting_card_spend",
    "avg_greeting_card_spend_per_trip", "avg_days_between_greeting_card_buys", "greeting_card_trips_count"
]

for col in numeric_cols:
    h2o_train[col] = h2o_train[col].asnumeric()
    h2o_test[col] = h2o_test[col].asnumeric()

In [0]:
# h2o is inefficient with enum and string types, checking to see if conversion happened as expected
print(h2o_train.types)
h2o_train.describe()

# small subset for example ...
suspect_cols = ["total_net_spend", "total_trips", "total_fuel_points", 
                "holiday_2024_kpf_net_spend", "holiday_2024_kpf_trips"]
for col in suspect_cols:
    print(col, h2o_train[col].nlevels())

In [0]:
# Run AutoML excluding Target Label and EHHN
target_col = "target_label" 
id_col = "ehhn"

h2o_train[target_col] = h2o_train[target_col].asfactor()
feature_cols = [col for col in h2o_train.columns if col not in [target_col, id_col]]

# Reduced max_models and cross-validation folds to prevent OOM
auto_ml = H2OAutoML(
    max_models=10, 
    nfolds=3, # 3-fold Cross-Validation
    seed=8451, 
    exclude_algos=['StackedEnsemble', 'DeepLearning'])


auto_ml.train(
    x=feature_cols, 
    y=target_col, 
    training_frame=h2o_train
)

h2o.cluster_status()


In [0]:
#h2o.cluster_status()

In [0]:
top_model = auto_ml.get_best_model()

In [0]:
# display metrics for top model
top_model.model_performance(h2o_test)

# AUC 

In [0]:
# View leaderboard
leaderboard = auto_ml.leaderboard
leaderboard.head()

# Good AUC, able to distinguish between classes relatively well. Curious to see what features led to this

# rmse, mse: avg of sqd differences between truth and prediction .. but i think this model is very good at
# identifying "0" label HHs (non-gift), which makes the mse number low 

gains_lift = top_model.gains_lift(h2o_test)
print(gains_lift)
# gains_lift: how many times better than random selection that group performs

In [0]:
# SHAP for feature importance + direction
#contributions = top_model.predict_contributions(h2o_test)
#contributions_df = contributions.as_data_frame()

h2o_test_sample = h2o_test.split_frame(ratios=[0.30], seed=8451)[0]
top_model.shap_summary_plot(h2o_test_sample)


In [0]:
# Prediction on h2o_test, not h2o_train 
preds = top_model.predict(h2o_test)

# Get HHs and their likelihood scores + classification
scored = h2o_test["ehhn"].cbind(preds)
scored.columns = ["ehhn", "predict", "p0", "p1"]
scored.head(20)

In [0]:
scored.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2025_predicted_HHS_result')

#